In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from fundamentos_procesamiento_imagenes.tarea_2.pregunta_2.procesamiento import (
    add_gaussian_noise, coefficient_map, derivatives, diffuse, load_image,
    neighbors, rmse, stability_limit
)

image = load_image("img.jpg")
noise_image = add_gaussian_noise(image, sigma=0.05, seed=0)

print(f"Dimensiones: {image.shape}")
print(f"RMSE de la imagen con ruido: {rmse(image, noise_image):.5f}")


In [ ]:
# Imagen de referencia y realizacion de ruido Gaussiano
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, data, title in zip(
    axes,
    (image, noise_image),
    ("Referencia en grises", "Ruido Gaussiano: desviación 0.05")
):
    ax.imshow(data, cmap="gray", vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
epsilons = [0.02, 0.05, 0.10, 0.20]
tv_results = {}
fig, axes = plt.subplots(2, 4, figsize=(16, 7))

for j, epsilon in enumerate(epsilons):
    lambda_ = 0.20 * epsilon
    iterations = max(1, round(0.04 / lambda_))
    c = coefficient_map(noise_image, "tv", epsilon=epsilon)
    result = diffuse(noise_image, "tv", lambda_, iterations, epsilon=epsilon)
    tv_results[epsilon] = result

    im = axes[0, j].imshow(c, cmap="viridis", vmin=0, vmax=1/epsilon)
    axes[0, j].set_title(f"cTV inicial, ε={epsilon}")
    axes[0, j].axis("off")
    fig.colorbar(im, ax=axes[0, j], fraction=0.046)

    axes[1, j].imshow(result, cmap="gray", vmin=0, vmax=1)
    axes[1, j].set_title(f"λ={lambda_:.3f}, n={iterations}\nRMSE={rmse(image, result):.4f}")
    axes[1, j].axis("off")

    print(f"TV ε={epsilon:.2f}: cmax real={c.max():.3f}, "
          f"cota cmax={1/epsilon:.1f}, λmax={stability_limit('tv', epsilon):.4f}, "
          f"RMSE={rmse(image, result):.5f}")

plt.tight_layout()
plt.show()


In [ ]:
tv_cases = [
    (0.05, 0.0005, 4, "Evolución lenta"),
    (0.05, 0.0100, 4, "Tiempo intermedio"),
    (0.05, 0.0100, 40, "Sobre-suavizado"),
]
fig, axes = plt.subplots(1, len(tv_cases), figsize=(15, 4))

for ax, (epsilon, lambda_, iterations, name) in zip(axes, tv_cases):
    result = diffuse(noise_image, "tv", lambda_, iterations, epsilon=epsilon)
    ax.imshow(result, cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"{name}\nε={epsilon}, λ={lambda_}, n={iterations}\n"
                 f"RMSE={rmse(image, result):.4f}")
    ax.axis("off")
    print(name, "tiempo", lambda_ * iterations, "RMSE", rmse(image, result))

plt.tight_layout()
plt.show()


In [ ]:
ks_borde = [0.04, 0.06, 0.08]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for j, k in enumerate(ks_borde):
    c = coefficient_map(noise_image, "borde", k=k)
    result = diffuse(noise_image, "borde", 0.20, 5, k=k)
    im = axes[0, j].imshow(c, cmap="viridis", vmin=0, vmax=1)
    axes[0, j].set_title(f"c inicial, k={k}")
    axes[0, j].axis("off")
    fig.colorbar(im, ax=axes[0, j], fraction=0.046)
    axes[1, j].imshow(result, cmap="gray", vmin=0, vmax=1)
    axes[1, j].set_title(f"λ=0.20, n=5\nRMSE={rmse(image, result):.4f}")
    axes[1, j].axis("off")
    print(f"Borde k={k:.3f}: c mediana={np.median(c):.3f}, "
          f"RMSE={rmse(image, result):.5f}")

plt.tight_layout()
plt.show()


In [ ]:
ks_lap = [0.015, 0.025, 0.040]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for j, k in enumerate(ks_lap):
    c = coefficient_map(noise_image, "laplaciano", k=k, alpha=0.5, sigma_pre=0.7)
    result = diffuse(noise_image, "laplaciano", 0.20, 5,
                     k=k, alpha=0.5, sigma_pre=0.7)
    im = axes[0, j].imshow(c, cmap="viridis", vmin=0, vmax=1)
    axes[0, j].set_title(f"c inicial, k={k}")
    axes[0, j].axis("off")
    fig.colorbar(im, ax=axes[0, j], fraction=0.046)
    axes[1, j].imshow(result, cmap="gray", vmin=0, vmax=1)
    axes[1, j].set_title(f"α=0.5, σpre=0.7, λ=0.20, n=5\n"
                         f"RMSE={rmse(image, result):.4f}")
    axes[1, j].axis("off")
    print(f"Laplaciano k={k:.3f}: c mediana={np.median(c):.3f}, "
          f"RMSE={rmse(image, result):.5f}")

plt.tight_layout()
plt.show()


In [ ]:
alphas = [0.0, 0.5, 1.0]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for j, alpha in enumerate(alphas):
    c = coefficient_map(noise_image, "laplaciano",
                        k=0.025, alpha=alpha, sigma_pre=0.7)
    result = diffuse(noise_image, "laplaciano", 0.20, 5,
                     k=0.025, alpha=alpha, sigma_pre=0.7)
    im = axes[0, j].imshow(c, cmap="viridis", vmin=0, vmax=1)
    axes[0, j].set_title(f"c inicial, α={alpha}")
    axes[0, j].axis("off")
    fig.colorbar(im, ax=axes[0, j], fraction=0.046)
    axes[1, j].imshow(result, cmap="gray", vmin=0, vmax=1)
    axes[1, j].set_title(f"k=0.025, σpre=0.7, λ=0.20, n=5\n"
                         f"RMSE={rmse(image, result):.4f}")
    axes[1, j].axis("off")
    print(f"α={alpha:.1f}: c mediana={np.median(c):.3f}, "
          f"RMSE={rmse(image, result):.5f}")

plt.tight_layout()
plt.show()


In [ ]:
pre_sigmas = [0.0, 0.4, 0.7, 1.0]
fig, axes = plt.subplots(2, 4, figsize=(16, 7))

for j, sigma_pre in enumerate(pre_sigmas):
    c = coefficient_map(noise_image, "laplaciano",
                        k=0.025, alpha=0.5, sigma_pre=sigma_pre)
    result = diffuse(noise_image, "laplaciano", 0.20, 5,
                     k=0.025, alpha=0.5, sigma_pre=sigma_pre)
    im = axes[0, j].imshow(c, cmap="viridis", vmin=0, vmax=1)
    axes[0, j].set_title(f"σpre={sigma_pre}")
    axes[0, j].axis("off")
    fig.colorbar(im, ax=axes[0, j], fraction=0.046)
    axes[1, j].imshow(result, cmap="gray", vmin=0, vmax=1)
    axes[1, j].set_title(f"RMSE={rmse(image, result):.4f}")
    axes[1, j].axis("off")
    print(f"σpre={sigma_pre:.1f}: c percentiles 5/50/95="
          f"{np.percentile(c, [5, 50, 95])}, RMSE={rmse(image, result):.5f}")

plt.tight_layout()
plt.show()


In [ ]:
parameters = {
    "tv": {"epsilon": 0.10},
    "borde": {"k": 0.08},
    "laplaciano": {"k": 0.025, "alpha": 0.5, "sigma_pre": 0.7},
}
lambda_values = {"tv": 0.02, "borde": 0.20, "laplaciano": 0.20}
iterations_values = {"tv": 2, "borde": 5, "laplaciano": 8}
results = {}
regions = {
    "Cielo": (slice(10, 75), slice(200, 450)),
    "Borde bosque": (slice(120, 270), slice(420, 565)),
    "Textura vegetal": (slice(320, 400), slice(150, 600)),
}

for method in parameters:
    results[method] = diffuse(
        noise_image, method, lambda_values[method], iterations_values[method],
        **parameters[method]
    )

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, data, title in zip(
    axes,
    (image, noise_image, results["tv"], results["borde"], results["laplaciano"]),
    ("Referencia", "Ruidosa", "TV", "Borde", "Laplaciano")
):
    ax.imshow(data, cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"{title}\nRMSE={rmse(image, data):.4f}")
    ax.axis("off")
plt.tight_layout()
plt.show()

print("RMSE global y por recorte (fila, columna)")
for name, data in (("Ruidosa", noise_image), *results.items()):
    scores = [rmse(image, data)]
    scores += [rmse(image[roi], data[roi]) for roi in regions.values()]
    print(f"{name:12s} " + " ".join(f"{score:.5f}" for score in scores))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, method in zip(axes, parameters):
    c = coefficient_map(noise_image, method, **parameters[method])
    im = ax.imshow(c, cmap="viridis")
    ax.set_title(f"{method}: c inicial")
    ax.axis("off")
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()


In [ ]:
roi = (slice(120, 270), slice(420, 565))
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, data, title in zip(
    axes,
    (image, noise_image, results["tv"], results["borde"], results["laplaciano"]),
    ("Referencia", "Ruidosa", "TV", "Borde", "Laplaciano")
):
    ax.imshow(data[roi], cmap="gray", vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

row = 200
cols = np.arange(420, 566)
plt.figure(figsize=(11, 4))
plt.plot(cols, image[row, cols], label="Referencia", linewidth=2)
plt.plot(cols, noise_image[row, cols], label="Ruidosa", alpha=0.5)
for method, result in results.items():
    plt.plot(cols, result[row, cols], label=method)
plt.xlabel("Columna (fila 200)")
plt.ylabel("Intensidad")
plt.title("Perfil sobre una frontera bosque/pradera")
plt.grid()
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
gradient, _ = derivatives(image)
roi_gradient = gradient[90:240, 430:580]
local_y, local_x = np.unravel_index(np.argmax(roi_gradient), roi_gradient.shape)
edge_pixel = (int(local_y + 90), int(local_x + 430))
c = coefficient_map(noise_image, "tv", epsilon=0.10)
un, us, uw, ue = neighbors(noise_image)
cn, cs, cw, ce = neighbors(c)

for name, (y, x) in (("Cielo", (35, 350)), ("Borde", edge_pixel)):
    differences = np.array([un[y, x], us[y, x], uw[y, x], ue[y, x]]) - noise_image[y, x]
    weights = (np.array([cn[y, x], cs[y, x], cw[y, x], ce[y, x]]) + c[y, x]) / 2
    change = 0.02 * np.sum(weights * differences)
    print(f"{name}, píxel (fila={y}, columna={x}):")
    print(f"  u^0={noise_image[y, x]:.6f}, c={c[y, x]:.6f}")
    print(f"  diferencias N/S/O/E={np.round(differences, 6)}")
    print(f"  coeficientes de cara N/S/O/E={np.round(weights, 6)}")
    print(f"  cambio={change:.6f}, u^1={noise_image[y, x] + change:.6f}")
